# 01. 속성별 감성 분류 모델 학습

`klue/roberta-base`를 긍정·부정·중립 3개 클래스로 파인튜닝
흐름: 데이터 확인 → 토큰화 → 학습 → Test 평가 → 저장

In [ ]:
# 라이브러리·프로젝트 경로·기본 설정
# 현재 폴더와 상위 폴더에서 aspect_labels.json을 찾아 프로젝트 위치 결정
# 로컬과 SSH 서버에서 같은 코드 사용
# MODEL_NAME은 한국어 사전학습 모델
# MAX_LENGTH는 한 입력에서 사용할 최대 토큰 수
# SEED는 다시 실행할 때 결과 차이를 줄이는 기준값
# DATA_DIR은 감성 학습 CSV 위치
# OUTPUT_DIR은 최종 감성 모델 저장 위치
# CHECKPOINT_DIR은 학습 중간 모델 임시 저장 위치
# GPU 사용 가능하면 GPU 사용, 아니면 CPU 사용

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

root_candidates = [Path.cwd(), *Path.cwd().parents, Path("/home/bteam/aspect_sentiment")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "aspect_labels.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("aspect_labels.json을 찾을 수 없습니다. 프로젝트 폴더 안에서 실행하세요.")

MODEL_NAME = "klue/roberta-base"
MAX_LENGTH = 64
SEED = 42
DATA_DIR = PROJECT_ROOT / "data" / "sentiment"
OUTPUT_DIR = PROJECT_ROOT / "models" / "sentiment"
CHECKPOINT_DIR = Path("/tmp/aspect_sentiment_checkpoints/sentiment")

set_seed(SEED)
print("프로젝트:", PROJECT_ROOT)
print("사용 장치:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Train·Validation·Test 데이터 읽기와 모델 입력 생성
# Train은 실제 공부, Validation은 학습 중 모델 선택, Test는 최종 시험에 사용
# aspect, sentiment_text, sentiment_polarity 필수 열이 없으면 바로 중단
# 속성과 문장의 앞뒤 공백 및 반복 공백 정리
# 원본 1은 학습용 0 긍정, 원본 -1은 학습용 1 부정, 원본 0은 학습용 2 중립
# 속성과 문장을 [속성] 속성명 [문장] 문장 형식으로 결합
# Train·Validation·Test 사이에 같은 모델 입력이 있으면 바로 중단
# 마지막 summary는 데이터 수, 감성별 개수, 카테고리 수 확인용

data = {
    split: pd.read_csv(DATA_DIR / f"{split}.csv")
    for split in ("train", "validation", "test")
}

# 원본 정답을 학습 번호로 변경
required = {"aspect", "sentiment_text", "sentiment_polarity"}
label_map = {1: 0, -1: 1, 0: 2}

for split, frame in data.items():
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{split} 필수 열 없음: {sorted(missing)}")

    for column in ("aspect", "sentiment_text"):
        frame[column] = (
            frame[column].fillna("").astype(str)
            .str.strip().str.replace(r"\s+", " ", regex=True)
        )
    frame["sentiment_polarity"] = frame["sentiment_polarity"].astype(int)
    unknown = sorted(set(frame["sentiment_polarity"]) - set(label_map))
    if unknown:
        raise ValueError(f"{split}에 정의되지 않은 감성값 존재: {unknown}")
    frame["label"] = frame["sentiment_polarity"].map(label_map).astype(int)
    frame["model_input"] = "[속성] " + frame["aspect"] + " [문장] " + frame["sentiment_text"]

# 전처리가 올바르게 되었는지 확인: 분할 간 같은 모델 입력은 허용하지 않음
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    overlap = set(data[left]["model_input"]) & set(data[right]["model_input"])
    if overlap:
        raise ValueError(f"{left}-{right} 사이 중복 입력 {len(overlap)}개: 전처리를 다시 실행하세요.")

summary = pd.DataFrame({
    split: {
        "행 수": len(frame),
        "긍정": int((frame["label"] == 0).sum()),
        "부정": int((frame["label"] == 1).sum()),
        "중립": int((frame["label"] == 2).sum()),
        "카테고리 수": frame["category"].nunique() if "category" in frame else None,
    }
    for split, frame in data.items()
}).T
display(summary)

In [ ]:
# 토큰화·학습용 Dataset·감성 모델 생성
# 토크나이저는 한글 문장을 KLUE-RoBERTa가 읽는 토큰 번호로 변환
# input_ids는 문장을 숫자로 바꾼 목록
# attention_mask는 실제 토큰 위치 1, 빈 padding 위치 0
# 64토큰을 넘는 입력은 뒤쪽 제거
# pandas 표에서 model_input과 label만 선택
# Train·Validation·Test에 같은 토큰화 함수 적용
# batched=True는 여러 문장을 묶어서 빠르게 토큰화
# 토큰화 후 학습용 Dataset에서 원래 문자열 열 제거
# 원본 CSV의 문자열은 그대로 유지
# 출력 번호 0은 긍정, 1은 부정, 2는 중립
# num_labels=3은 긍정·부정·중립 logits 3개 출력

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 문장을 토큰으로 나누고 토큰 번호로 바꿈
def tokenize(batch):
    return tokenizer(batch["model_input"], truncation=True, max_length=MAX_LENGTH)
# 토큰화를 데이터 전체에 적용
datasets = {}
for split, frame in data.items():   # model_input(속성+문장)과 라벨을 가져옴
    source = Dataset.from_pandas(frame[["model_input", "label"]], preserve_index=False)
    datasets[split] = source.map(
        tokenize,
        batched=True,
        remove_columns=["model_input"],
    )

id2label = {0: "긍정", 1: "부정", 2: "중립"}
label2id = {name: number for number, name in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
# 클래스 가중치·손실 함수·평가 지표
# 감성별 Train 데이터 수가 달라 balanced 클래스 가중치 사용
# 데이터가 많은 긍정은 작은 가중치, 데이터가 적은 중립은 큰 가중치
# 적은 감성을 모델이 무시하지 않도록 보정
# outputs.logits는 모델이 출력한 긍정·부정·중립 원점수
# labels는 실제 정답 번호
# cross_entropy는 예측과 정답의 차이를 loss로 계산
# 학습은 loss가 작아지는 방향으로 모델 내부 가중치 수정
# accuracy는 전체 정답률
# macro_f1은 긍정·부정·중립을 같은 비중으로 계산한 성능
# positive_f1, negative_f1, neutral_f1은 감성별 성능
# 확률 표시가 필요 없으므로 logits에서 가장 큰 위치를 바로 예측으로 선택

classes = np.array([0, 1, 2])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=data["train"]["label"].to_numpy(),
)
class_weights = torch.tensor(weights, dtype=torch.float)
print("클래스 가중치:", dict(zip(id2label.values(), weights.round(4))))

def weighted_loss(outputs, labels, num_items_in_batch=None):
    return F.cross_entropy(
        outputs.logits,
        labels,
        weight=class_weights.to(outputs.logits.device),
    )

def compute_metrics(result):
    logits, answers = result
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = precision_recall_fscore_support(
        answers, predictions, average="macro", zero_division=0
    )[2]
    class_f1 = precision_recall_fscore_support(
        answers, predictions, labels=[0, 1, 2], average=None, zero_division=0
    )[2]
    return {
        "accuracy": accuracy_score(answers, predictions),
        "macro_f1": macro_f1,
        "positive_f1": class_f1[0],
        "negative_f1": class_f1[1],
        "neutral_f1": class_f1[2],
    }

In [ ]:
# 학습 조건 설정·Trainer 생성·실제 파인튜닝
# Train 전체를 3번 반복
# learning_rate는 모델 내부 값을 한 번에 수정하는 크기
# 학습은 문장 16개씩, 평가는 문장 32개씩 처리
# epoch가 끝날 때마다 Validation 평가와 중간 모델 저장
# Validation Macro F1이 가장 높은 모델을 최고 모델로 선택
# 학습 종료 후 마지막 모델이 아니라 최고 모델 복원
# DataCollatorWithPadding은 한 batch 안의 문장 길이를 자동으로 맞춤
# trainer.train()에서 실제 학습 시작
# 문장 입력 → logits → loss → 역전파 → 가중치 수정 순서로 반복

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    compute_loss_func=weighted_loss,
)

trainer.train()

In [ ]:
# 최고 모델 저장·Validation/Test 최종 평가
# 최고 모델과 토크나이저를 models/sentiment에 저장
# 서비스에서는 이 폴더를 불러와 감성 예측
# Validation은 최고 모델 선택에 사용한 모의고사
# Test는 모델 선택이 끝난 뒤 한 번 확인하는 최종 시험
# metrics.csv는 Validation·Test 전체 성능
# classification_report.csv는 긍정·부정·중립별 성능
# test_wrong_predictions.csv는 Test에서 틀린 문장과 예측값
# 오답 파일은 반전 표현, 문맥 잘림, 중립 오류, 라벨 문제 확인용

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

validation_metrics = trainer.evaluate(
    datasets["validation"],
    metric_key_prefix="validation",
)
test_output = trainer.predict(datasets["test"], metric_key_prefix="test")
test_predictions = np.argmax(test_output.predictions, axis=-1)

metrics = {
    "best_validation_macro_f1": trainer.state.best_metric,
    **validation_metrics,
    **test_output.metrics,
}
metrics_df = pd.DataFrame(metrics.items(), columns=["metric", "value"])
metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False, encoding="utf-8-sig")

report = classification_report(
    data["test"]["label"],
    test_predictions,
    labels=[0, 1, 2],
    target_names=[id2label[i] for i in range(3)],
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).transpose().to_csv(
    OUTPUT_DIR / "classification_report.csv",
    encoding="utf-8-sig",
)

test_result = data["test"].copy()
test_result["predicted_label"] = test_predictions
test_result["predicted_sentiment"] = test_result["predicted_label"].map(id2label)
wrong = test_result[test_result["label"] != test_result["predicted_label"]]
wrong.to_csv(
    OUTPUT_DIR / "test_wrong_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

display(metrics_df)
print("완료:", OUTPUT_DIR)
print("Test 오답:", len(wrong))